In [0]:
from pyspark import pipelines as dp

In [0]:
from pyspark.sql.functions import col,countDistinct,sum,avg,count,when

In [0]:
@dp.materialized_view(
    name = "restaurant.gold.d_restaurant_reviews",
    table_properties={"quality":"gold"},
    comment = "this is restaurant review table"
)
def restaurant_reviews():
    df_review = spark.read.table("restaurant.silver.fact_reviews")
    df_rest = spark.table("restaurant.silver.dim_restaurants")
    df_joined = df_review.alias("re").join(df_rest.alias("rs"),"restaurant_id","inner") \
                                    .groupBy(col("rs.restaurant_id"),col("rs.name"),col("rs.city")) \
                                    .agg(count("re.review_id").alias("total_reviews"),avg(col("re.rating")).alias("avg_rating").cast("decimal (3,2)"),count(when(col("rating") == 5,col("re.review_id")).otherwise(None)).alias("rating_5_count"),count(when(col("rating") == 4,col("re.review_id")).otherwise(None)).alias("rating_4_count"),count(when(col("rating") == 3,col("re.review_id")).otherwise(None)).alias("rating_3_count"),count(when(col("rating") == 2,col("re.review_id")).otherwise(None)).alias("rating_2_count"),count(when(col("rating") == 1,col("re.review_id")).otherwise(None)).alias("rating_1_count"),count(when(col("sentiment") == "positive",col("review_id")).otherwise(None)).alias("sentiment_postive_count"),count(when(col("sentiment") == "negative",col("review_id")).otherwise(None)).alias("sentiment_negative_count"),count(when(col("sentiment") == "neutral",col("review_id")).otherwise(None)).alias("sentiment_neutral_count"))
    return df_joined